# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pushpendrajat2004/FlyRank01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds a **transparent, rule-based baseline** that every future model must beat.
The rule scores pages by visibility, freshness risk, position opportunity, and content depth —
then attaches reason codes so a human can audit every pick.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ---- Setup: load data, prepare features ----
import sys, pathlib
import numpy as np
import pandas as pd

# Resolve repo root regardless of where the notebook is run
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError('Could not find repo root with data/raw/content_refresh_anonymized.csv')
    REPO_ROOT = REPO_ROOT.parent

RAW_CSV = REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
OUTPUT_DIR = REPO_ROOT / 'work' / 'outputs'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_CSV)
print(f'Raw CSV: {len(df_raw):,} rows × {df_raw.shape[1]} columns')

Raw CSV: 30,000 rows × 44 columns


In [2]:
# ---- Minimal feature prep (mirrors scripts/01_prepare_features.py) ----
NUMERIC_FILL_ZERO = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'age_tier_order', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'ai_traffic_pct', 'trend_pct',
]

df = df_raw.copy()
for col in NUMERIC_FILL_ZERO:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

CATEGORICAL_FILL = [
    'competition_level', 'content_type', 'main_intent',
    'provider_used', 'model_used', 'age_tier', 'freshness_tier',
    'word_count_tier', 'char_count_tier', 'impression_tier',
    'position_tier', 'trend_direction',
]
for col in CATEGORICAL_FILL:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str).replace({'': 'unknown', 'nan': 'unknown'})

# Keep only pages with traffic and sufficient age
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset=['content_id']).reset_index(drop=True)

# The label
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

print(f'Prepared: {len(df):,} rows')
print(f'Declining base rate: {df["is_declining_label"].mean():.3f} ({df["is_declining_label"].sum():,} / {len(df):,})')

Prepared: 30,000 rows
Declining base rate: 0.542 (16,262 / 30,000)


## 1. Signal checks — do the signals my rule leans on actually separate decliners?

My rule is built on four signals: **visibility** (impressions), **staleness** (days since update),
**position opportunity** (avg_position), and **depth gap** (word_count).

Before encoding the rule, I need to verify that these signals actually correlate with declining pages.
I'll check two:

1. **Staleness** (flag-linked: directly behind FlyRank's refresh flags) — do staler pages decline more?
2. **CTR vs Position** (flag-linked: behind the CTR-fix logic) — do pages with low CTR for their position decline more?

In [3]:
# ---- Signal Check 1: STALENESS (days_since_last_update) ----
# This is behind FlyRank's refresh flags: staler pages should need refreshing more.
# Question: Do pages that haven't been updated in a long time decline at a higher rate?

staleness_bins = [0, 30, 90, 180, 365, float('inf')]
staleness_labels = ['0-30d', '31-90d', '91-180d', '181-365d', '365d+']

df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=staleness_bins,
    labels=staleness_labels,
    right=True
)

staleness_table = df.groupby('staleness_bucket', observed=True).agg(
    n=('is_declining_label', 'count'),
    n_declining=('is_declining_label', 'sum'),
    decline_rate=('is_declining_label', 'mean'),
    median_impressions=('impressions_90d', 'median'),
).round(3)

overall_rate = df['is_declining_label'].mean()
staleness_table['lift_vs_base'] = (staleness_table['decline_rate'] / overall_rate).round(2)

print('=== Signal Check 1: STALENESS (days_since_last_update) ===')
print(f'Overall declining rate (base): {overall_rate:.3f}')
print(f'Total n: {len(df):,}')
print()
print(staleness_table.to_string())
print()

# Verdict logic: check if there is a monotonic trend
rates = staleness_table['decline_rate'].values
if all(rates[i] <= rates[i+1] for i in range(len(rates)-1)):
    staleness_verdict = 'CONFIRMED'
    print(f'Verdict: {staleness_verdict}')
    print('Decline rate rises monotonically with staleness — staler pages decline more often.')
elif rates[-1] > rates[0] and (rates[-1] - rates[0]) > 0.05:
    staleness_verdict = 'CONFIRMED'
    print(f'Verdict: {staleness_verdict}')
    print('Decline rate is directionally higher for staler pages (not perfectly monotonic, but clear signal).')
elif abs(rates[-1] - rates[0]) < 0.03:
    staleness_verdict = 'FALSE'
    print(f'Verdict: {staleness_verdict}')
    print('No meaningful difference in decline rate across staleness buckets.')
else:
    staleness_verdict = 'MIXED'
    print(f'Verdict: {staleness_verdict}')
    print('Signal is noisy — some buckets higher, some lower, no clear monotonic pattern.')

=== Signal Check 1: STALENESS (days_since_last_update) ===
Overall declining rate (base): 0.542
Total n: 30,000

                      n  n_declining  decline_rate  median_impressions  lift_vs_base
staleness_bucket                                                                    
0-30d             20480        10473         0.511               470.0          0.94
31-90d              175          103         0.589               510.0          1.09
91-180d            9171         5604         0.611              1692.0          1.13
181-365d            169           79         0.467                16.0          0.86
365d+                 5            3         0.600                 2.0          1.11

Verdict: CONFIRMED
Decline rate is directionally higher for staler pages (not perfectly monotonic, but clear signal).


In [4]:
# ---- Signal Check 2: CTR vs POSITION (flag-linked: behind CTR-fix logic) ----
# FlyRank flags pages with low CTR for their position — the idea is that a page
# ranking well but not getting clicks is underperforming and likely to decline.
#
# Question: Do pages with low CTR relative to their position bucket decline more?

# Filter to pages with valid position data (avg_position > 0)
df_pos = df[df['avg_position'] > 0].copy()
print(f'Pages with valid position data: n = {len(df_pos):,} (excluded {len(df) - len(df_pos):,} with avg_position=0)')
print()

# Create position tiers
pos_bins = [0, 3, 10, 20, 50, float('inf')]
pos_labels = ['top_3', 'page_1 (4-10)', 'striking (11-20)', 'page_3-5 (21-50)', 'deep (50+)']
df_pos['pos_bucket'] = pd.cut(
    df_pos['avg_position'],
    bins=pos_bins,
    labels=pos_labels,
    right=True
)

# Within each position bucket, split by CTR relative to bucket median
# Remember: ctr is ×100, so 0.76 means 0.76%
bucket_median_ctr = df_pos.groupby('pos_bucket', observed=True)['ctr'].transform('median')
df_pos['ctr_vs_bucket'] = np.where(df_pos['ctr'] < bucket_median_ctr, 'below_median_CTR', 'at_or_above_median_CTR')

ctr_pos_table = df_pos.groupby(['pos_bucket', 'ctr_vs_bucket'], observed=True).agg(
    n=('is_declining_label', 'count'),
    n_declining=('is_declining_label', 'sum'),
    decline_rate=('is_declining_label', 'mean'),
    median_ctr=('ctr', 'median'),
    median_impressions=('impressions_90d', 'median'),
).round(3)

print('=== Signal Check 2: CTR vs POSITION ===')
print(f'Overall declining rate (base): {overall_rate:.3f}')
print()
print(ctr_pos_table.to_string())
print()

# Check: in each position bucket, does below-median CTR have a higher decline rate?
n_buckets_checked = 0
n_ctr_signal_holds = 0
for pos_tier in pos_labels:
    try:
        below = ctr_pos_table.loc[(pos_tier, 'below_median_CTR'), 'decline_rate']
        above = ctr_pos_table.loc[(pos_tier, 'at_or_above_median_CTR'), 'decline_rate']
        n_buckets_checked += 1
        if below > above:
            n_ctr_signal_holds += 1
    except KeyError:
        pass

if n_buckets_checked == 0:
    ctr_verdict = 'FALSE'
    print(f'Verdict: {ctr_verdict}')
    print('Could not evaluate — not enough data in position buckets.')
elif n_ctr_signal_holds == n_buckets_checked:
    ctr_verdict = 'CONFIRMED'
    print(f'Verdict: {ctr_verdict}')
    print(f'In all {n_buckets_checked} position tiers, below-median CTR pages decline more.')
elif n_ctr_signal_holds >= n_buckets_checked * 0.6:
    ctr_verdict = 'CONFIRMED'
    print(f'Verdict: {ctr_verdict}')
    print(f'In {n_ctr_signal_holds}/{n_buckets_checked} position tiers, below-median CTR pages decline more — signal holds directionally.')
elif n_ctr_signal_holds >= n_buckets_checked * 0.4:
    ctr_verdict = 'MIXED'
    print(f'Verdict: {ctr_verdict}')
    print(f'In {n_ctr_signal_holds}/{n_buckets_checked} position tiers the signal holds — inconsistent.')
else:
    ctr_verdict = 'OPPOSITE'
    print(f'Verdict: {ctr_verdict}')
    print(f'In only {n_ctr_signal_holds}/{n_buckets_checked} position tiers the signal holds — reversed direction.')

Pages with valid position data: n = 28,795 (excluded 1,205 with avg_position=0)

=== Signal Check 2: CTR vs POSITION ===
Overall declining rate (base): 0.542

                                            n  n_declining  decline_rate  median_ctr  median_impressions
pos_bucket       ctr_vs_bucket                                                                          
top_3            at_or_above_median_CTR  1141          568         0.498        0.00                74.0
page_1 (4-10)    at_or_above_median_CTR  5955         3195         0.537        0.41              3233.0
                 below_median_CTR        5887         3548         0.603        0.00               159.0
striking (11-20) at_or_above_median_CTR  3760         2164         0.576        0.29              1838.5
                 below_median_CTR        3513         2269         0.646        0.00               253.0
page_3-5 (21-50) at_or_above_median_CTR  3664         2195         0.599        0.16              2634.5
 

In [5]:
# ---- Signal verdicts summary ----
print('╔══════════════════════════════════════════════════════════╗')
print('║              SIGNAL VERDICTS SUMMARY                    ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  1. Staleness (days_since_last_update)  →  {staleness_verdict:<12s} ║')
print(f'║     Flag-linked: refresh flags                         ║')
print(f'║  2. CTR vs Position                     →  {ctr_verdict:<12s} ║')
print(f'║     Flag-linked: CTR-fix logic                         ║')
print('╚══════════════════════════════════════════════════════════╝')

# Clean up temp columns
df.drop(columns=['staleness_bucket'], inplace=True, errors='ignore')

╔══════════════════════════════════════════════════════════╗
║              SIGNAL VERDICTS SUMMARY                    ║
╠══════════════════════════════════════════════════════════╣
║  1. Staleness (days_since_last_update)  →  CONFIRMED    ║
║     Flag-linked: refresh flags                         ║
║  2. CTR vs Position                     →  CONFIRMED    ║
║     Flag-linked: CTR-fix logic                         ║
╚══════════════════════════════════════════════════════════╝


## 1b. My rule and its reason codes

**The rule in plain words:**

> A page is worth reviewing if it used to get traffic (visibility), it is getting old without
> an update (freshness risk), its position offers room for improvement (position opportunity),
> and its content is thin relative to its visibility (depth gap).

The score is a weighted sum of four percentile-ranked signals:

| Component | Weight | Meaning |
|---|---|---|
| `visibility_score` | 0.40 | log-impressions percentile — pages with more search demand get priority |
| `freshness_risk_score` | 0.30 | days-since-update percentile — staler pages score higher |
| `position_opportunity_score` | 0.25 | inverse-position × visibility — pages near page 1 with traffic get a boost |
| `depth_gap_score` | 0.05 | inverse-word-count × visibility — thin but visible pages surface |

**Reason codes** (each row carries WHY it scored):

| Code | Trigger |
|---|---|
| `stale_visible_page` | ≥ 180 days since update AND ≥ 500 impressions |
| `thin_visible_page` | word_count < 1,200 AND ≥ 250 impressions |
| `page_one_decay_risk` | avg_position ≤ 10 AND age ≥ 180 days |
| `low_ctr_visible_page` | CTR < 0.5% AND position ≤ 20 AND ≥ 500 impressions |
| `low_engagement_visible_page` | engagement_rate or scroll_rate < 30 AND ≥ 30 sessions |
| `general_refresh_review` | fallback — no other code fired |

**Note:** `declining_with_demand` was removed from reason codes because `trend_direction` is the label source — using it in reason codes would leak the label into the ranked queue.

In [6]:
# ---- Helper functions (from scripts/ml_utils.py) ----
def normalize(series):
    values = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    mn, mx = values.min(), values.max()
    if not np.isfinite(mn) or not np.isfinite(mx) or mx == mn:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - mn) / (mx - mn)

def percentile_rank(series):
    values = pd.to_numeric(series, errors='coerce').fillna(0)
    return values.rank(method='average', pct=True).fillna(0)

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': list(y_true), 'score': list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values('score', ascending=False).head(min(k, len(frame)))
    return float(top['y'].mean()) if len(top) else 0.0

print('Helpers loaded.')

Helpers loaded.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

In [7]:
# ---- Build the four score components ----
df['visibility_score']          = percentile_rank(np.log1p(df['impressions_90d']))
df['freshness_risk_score']      = percentile_rank(df['days_since_last_update'])
df['position_opportunity_score'] = (
    (1 - normalize(df['avg_position'].clip(lower=1, upper=50)))
    * df['visibility_score']
    * (df['avg_position'] > 0).astype(int)   # avg_position=0 means "no data"
)
df['depth_gap_score'] = (1 - percentile_rank(df['word_count'])) * df['visibility_score']

# ---- Combine into baseline score ----
df['baseline_refresh_score'] = (
    0.40 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.25 * df['position_opportunity_score']
    + 0.05 * df['depth_gap_score']
).clip(0, 1)

print('Score distribution:')
print(df['baseline_refresh_score'].describe().round(4))

Score distribution:


count    30000.0000
mean         0.4489
std          0.2156
min          0.0080
25%          0.2814
50%          0.4429
75%          0.6233
max          0.9412
Name: baseline_refresh_score, dtype: float64


In [8]:
# ---- Reason codes ----
# NOTE: trend_direction / trend_pct are label sources — NOT used here.
def reason_codes(row):
    reasons = []
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        reasons.append('stale_visible_page')
    if row['word_count'] > 0 and row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        reasons.append('thin_visible_page')
    if row['avg_position'] > 0 and row['avg_position'] <= 10 and row['content_age_days'] >= 180:
        reasons.append('page_one_decay_risk')
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        reasons.append('low_ctr_visible_page')
    if row['sessions_90d'] >= 30 and (
        (row['engagement_rate'] > 0 and row['engagement_rate'] < 30)
        or (row['scroll_rate'] > 0 and row['scroll_rate'] < 30)
    ):
        reasons.append('low_engagement_visible_page')
    if not reasons:
        reasons.append('general_refresh_review')
    return '|'.join(reasons)

df['reason_codes'] = df.apply(reason_codes, axis=1)

# ---- Suggested action from reason codes ----
def suggested_action(row):
    reasons = set(str(row['reason_codes']).split('|'))
    if 'thin_visible_page' in reasons:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in reasons:
        return 'refresh_and_review_ctr'
    if 'stale_visible_page' in reasons:
        return 'refresh'
    return 'monitor'

df['suggested_action_baseline'] = df.apply(suggested_action, axis=1)

# ---- Rank ----
df['baseline_rank'] = df['baseline_refresh_score'].rank(method='first', ascending=False).astype(int)

# ---- Reason code frequency ----
all_reasons = df['reason_codes'].str.split('|').explode()
print('Reason code distribution:')
print(all_reasons.value_counts())
print(f'\nAction distribution:')
print(df['suggested_action_baseline'].value_counts())

Reason code distribution:
reason_codes
general_refresh_review         13644
low_ctr_visible_page            9759
page_one_decay_risk             7076
low_engagement_visible_page     6508
thin_visible_page                 82
stale_visible_page                17
Name: count, dtype: int64

Action distribution:
suggested_action_baseline
monitor                   20170
refresh_and_review_ctr     9741
expand_and_refresh           82
refresh                       7
Name: count, dtype: int64


In [9]:
# ---- Write the baseline CSV ----
output_cols = [
    'content_id', 'client_id', 'baseline_rank', 'baseline_refresh_score',
    'visibility_score', 'freshness_risk_score', 'position_opportunity_score',
    'depth_gap_score', 'reason_codes', 'suggested_action_baseline',
    'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction',
]

out = df[output_cols].sort_values('baseline_rank')
output_path = OUTPUT_DIR / 'baseline_action_score.csv'
out.to_csv(output_path, index=False)
print(f'Wrote {len(out):,} rows → {output_path}')

# ---- Precision@K evaluation ----
base_rate = df['is_declining_label'].mean()
for k in [10, 20, 50, 100]:
    pk = precision_at_k(df['is_declining_label'], df['baseline_refresh_score'], k)
    print(f'precision@{k:>3d} = {pk:.3f}   (base rate = {base_rate:.3f}, lift = {pk/base_rate:.2f}×)')

Wrote 30,000 rows → C:\Users\link4\OneDrive\Desktop\flyrank\FlyRank01\work\outputs\baseline_action_score.csv
precision@ 10 = 0.200   (base rate = 0.542, lift = 0.37×)
precision@ 20 = 0.350   (base rate = 0.542, lift = 0.65×)
precision@ 50 = 0.340   (base rate = 0.542, lift = 0.63×)
precision@100 = 0.380   (base rate = 0.542, lift = 0.70×)


In [10]:
# ---- Save metrics JSON (committable receipt) ----
import json

metrics = {
    'n_rows': int(len(df)),
    'base_rate': round(float(base_rate), 4),
    'signal_verdicts': {
        'staleness': staleness_verdict,
        'ctr_vs_position': ctr_verdict,
    },
    'precision_at_k': {},
    'score_inputs': ['impressions_90d', 'days_since_last_update', 'avg_position', 'word_count'],
}
for k in [10, 20, 50, 100]:
    pk = precision_at_k(df['is_declining_label'], df['baseline_refresh_score'], k)
    metrics['precision_at_k'][f'p@{k}'] = round(float(pk), 4)

metrics_path = OUTPUT_DIR / 'baseline_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Wrote metrics → {metrics_path}')
print(json.dumps(metrics, indent=2))

Wrote metrics → C:\Users\link4\OneDrive\Desktop\flyrank\FlyRank01\work\outputs\baseline_metrics.json
{
  "n_rows": 30000,
  "base_rate": 0.5421,
  "signal_verdicts": {
    "staleness": "CONFIRMED",
    "ctr_vs_position": "CONFIRMED"
  },
  "precision_at_k": {
    "p@10": 0.2,
    "p@20": 0.35,
    "p@50": 0.34,
    "p@100": 0.38
  },
  "score_inputs": [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "word_count"
  ]
}


## 3. Top-10 review (minimum) / Top-20 review

*For each of the top 10 (at minimum): action, reason code, confidence note, and what would make it wrong.*

In [11]:
# ---- Top-20 detail table ----
top20 = out.head(20).copy()

def confidence_note(row):
    """Quick heuristic confidence: high if volume + position data both exist,
    medium if one is weak, low if volume is tiny."""
    if row['impressions_90d'] >= 1000 and row['avg_position'] > 0:
        return 'HIGH — strong traffic + position data'
    if row['impressions_90d'] >= 300:
        return 'MEDIUM — moderate traffic'
    return 'LOW — thin traffic, noisy signal'

def what_would_make_it_wrong(row):
    """For each row, state the assumption that could invalidate this pick."""
    reasons = set(str(row['reason_codes']).split('|'))
    notes = []
    if 'stale_visible_page' in reasons:
        notes.append('Wrong if the page was intentionally left unchanged (e.g. evergreen FAQ).')
    if 'thin_visible_page' in reasons:
        notes.append('Wrong if the short length is intentional (e.g. comparison tables, tools).')
    if 'page_one_decay_risk' in reasons:
        notes.append('Wrong if page-1 position is stable and the age reflects maturity, not decay.')
    if 'low_ctr_visible_page' in reasons:
        notes.append('Wrong if CTR is structurally low for the SERP type (e.g. featured snippet steals clicks).')
    if 'low_engagement_visible_page' in reasons:
        notes.append('Wrong if low engagement is expected for the content type (e.g. quick-answer pages).')
    if not notes:
        notes.append('General review — wrong if the page has no realistic improvement path.')
    return ' '.join(notes)

top20['confidence'] = top20.apply(confidence_note, axis=1)
top20['wrong_if'] = top20.apply(what_would_make_it_wrong, axis=1)

# Display the review table
review_cols = [
    'baseline_rank', 'suggested_action_baseline', 'reason_codes',
    'baseline_refresh_score', 'confidence', 'wrong_if',
    'impressions_90d', 'avg_position', 'days_since_last_update',
    'word_count', 'ctr', 'is_declining_label',
]
pd.set_option('display.max_colwidth', 100)
print('=== TOP-10 REVIEW (required) ===')
display(top20[review_cols].head(10))
print()
print('=== ROWS 11-20 (bonus) ===')
display(top20[review_cols].tail(10))

=== TOP-10 REVIEW (required) ===


,baseline_rank,suggested_action_baseline,reason_codes,baseline_refresh_score,confidence,wrong_if,impressions_90d,avg_position,days_since_last_update,word_count,ctr,is_declining_label
21565,1,monitor,page_one_decay_risk|low_engagement_visible_page,0.941189,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",309192,2.0,104,0.0,0.87,1
4644,2,monitor,page_one_decay_risk|low_engagement_visible_page,0.934889,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",97999,2.5,104,0.0,0.52,0
18954,3,monitor,page_one_decay_risk|low_engagement_visible_page,0.934080,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",101078,2.7,104,0.0,0.85,0
17400,4,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,0.933606,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if CTR is str...",117741,3.0,104,0.0,0.45,0
9348,5,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,0.933559,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if CTR is str...",152617,3.3,104,0.0,0.29,0
25409,6,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,0.933263,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if CTR is str...",145292,3.3,104,0.0,0.46,1
18458,7,monitor,page_one_decay_risk|low_engagement_visible_page,0.932991,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",79146,2.5,104,0.0,0.73,0
13306,8,monitor,page_one_decay_risk|low_engagement_visible_page,0.931623,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",142072,3.6,104,0.0,0.83,0
28354,9,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,0.931363,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if CTR is str...",148737,3.7,104,0.0,0.48,0
8275,10,monitor,page_one_decay_risk|low_engagement_visible_page,0.931124,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",129239,3.6,104,0.0,0.55,0



=== ROWS 11-20 (bonus) ===


,baseline_rank,suggested_action_baseline,reason_codes,baseline_refresh_score,confidence,wrong_if,impressions_90d,avg_position,days_since_last_update,word_count,ctr,is_declining_label
2169,11,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,0.931033,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if CTR is str...",127907,3.6,104,0.0,0.34,0
19304,12,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,0.930401,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if CTR is str...",187893,4.0,104,0.0,0.45,1
6653,13,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,0.930217,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if CTR is str...",517715,4.2,104,0.0,0.14,1
10893,14,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,0.930125,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if CTR is str...",73675,2.9,104,0.0,0.19,0
16591,15,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,0.930118,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if CTR is str...",142505,3.9,104,0.0,0.29,1
13537,16,monitor,page_one_decay_risk|low_engagement_visible_page,0.930059,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",347399,4.2,104,0.0,0.53,1
16959,17,monitor,page_one_decay_risk|low_engagement_visible_page,0.930058,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",51233,2.0,104,0.0,0.88,0
26935,18,monitor,page_one_decay_risk|low_engagement_visible_page,0.929529,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",89311,3.4,104,0.0,1.01,0
4495,19,monitor,page_one_decay_risk|low_engagement_visible_page,0.929302,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",87433,3.4,104,0.0,0.89,0
17127,20,monitor,page_one_decay_risk|low_engagement_visible_page,0.929007,HIGH — strong traffic + position data,"Wrong if page-1 position is stable and the age reflects maturity, not decay. Wrong if low engage...",83603,3.4,104,0.0,1.06,1


In [12]:
# ---- Top-20 summary statistics ----
n_declining_top20 = top20['is_declining_label'].sum()
p_top20 = top20['is_declining_label'].mean()
n_declining_top10 = top20.head(10)['is_declining_label'].sum()
p_top10 = top20.head(10)['is_declining_label'].mean()

print(f'Top-10 declining: {n_declining_top10}/10 = {p_top10:.1%}')
print(f'Top-20 declining: {n_declining_top20}/20 = {p_top20:.1%}')
print(f'Base rate:        {base_rate:.1%}')
print(f'Lift over random (top-10): {p_top10/base_rate:.2f}×')
print(f'Lift over random (top-20): {p_top20/base_rate:.2f}×')
print()
print('Action breakdown in top 20:')
print(top20['suggested_action_baseline'].value_counts())
print()
print('Reason code breakdown in top 20:')
print(top20['reason_codes'].str.split('|').explode().value_counts())

Top-10 declining: 2/10 = 20.0%
Top-20 declining: 7/20 = 35.0%
Base rate:        54.2%
Lift over random (top-10): 0.37×
Lift over random (top-20): 0.65×

Action breakdown in top 20:
suggested_action_baseline
monitor                   11
refresh_and_review_ctr     9
Name: count, dtype: int64

Reason code breakdown in top 20:
reason_codes
page_one_decay_risk            20
low_engagement_visible_page    20
low_ctr_visible_page            9
Name: count, dtype: int64


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
# ---- Identify weak picks in the top 20 ----
# Weak pick: pages the rule flagged highly but that are NOT actually declining
weak_picks = top20[top20['is_declining_label'] == 0].copy()
print(f'Weak picks in top 20: {len(weak_picks)} pages flagged for action that are NOT declining.\n')

if len(weak_picks) > 0:
    print('Weak pick details:')
    display(weak_picks[[
        'baseline_rank', 'suggested_action_baseline', 'reason_codes',
        'impressions_90d', 'avg_position', 'days_since_last_update',
        'word_count', 'trend_direction',
    ]])
    print()
    print('Why these are weak: the rule surfaced them because of staleness or thin')
    print('content, but their traffic is actually stable or growing. The rule does not')
    print('distinguish between "stale and declining" vs "stale and stable" — that is the')
    print('gap a model should close.')
else:
    print('No weak picks found — look harder, this is unlikely with a rule-based score.')

Weak picks in top 20: 13 pages flagged for action that are NOT declining.

Weak pick details:


,baseline_rank,suggested_action_baseline,reason_codes,impressions_90d,avg_position,days_since_last_update,word_count,trend_direction
4644,2,monitor,page_one_decay_risk|low_engagement_visible_page,97999,2.5,104,0.0,stable
18954,3,monitor,page_one_decay_risk|low_engagement_visible_page,101078,2.7,104,0.0,stable
17400,4,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,117741,3.0,104,0.0,stable
9348,5,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,152617,3.3,104,0.0,stable
18458,7,monitor,page_one_decay_risk|low_engagement_visible_page,79146,2.5,104,0.0,stable
13306,8,monitor,page_one_decay_risk|low_engagement_visible_page,142072,3.6,104,0.0,stable
28354,9,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,148737,3.7,104,0.0,stable
8275,10,monitor,page_one_decay_risk|low_engagement_visible_page,129239,3.6,104,0.0,stable
2169,11,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,127907,3.6,104,0.0,stable
10893,14,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page,73675,2.9,104,0.0,stable



Why these are weak: the rule surfaced them because of staleness or thin
content, but their traffic is actually stable or growing. The rule does not
distinguish between "stale and declining" vs "stale and stable" — that is the
gap a model should close.


In [14]:
# ---- Leakage check ----
# 1. Confirm trend_direction and trend_pct are NOT used as score inputs
# The four score components use ONLY:
#   - impressions_90d  (visibility_score)
#   - days_since_last_update  (freshness_risk_score)
#   - avg_position  (position_opportunity_score)
#   - word_count  (depth_gap_score)
# None of these are derived from the label.

score_inputs = ['impressions_90d', 'days_since_last_update', 'avg_position', 'word_count']
label_sources = ['trend_direction', 'trend_pct', 'is_declining_label']

print('=== Leakage Check ===')
print()
print('Score inputs:', score_inputs)
print('Label sources (must NOT appear in score inputs):', label_sources)
overlap = set(score_inputs) & set(label_sources)
print(f'Overlap: {overlap if overlap else "NONE — CLEAN"}')
print()

# 2. Correlation sanity: score should correlate with the label moderately, not perfectly
score_label_corr = df['baseline_refresh_score'].corr(df['is_declining_label'])
print(f'Score ↔ label correlation: {score_label_corr:.4f}')
if abs(score_label_corr) > 0.8:
    print('⚠️ WARNING: suspiciously high correlation — investigate leakage!')
else:
    print('✓ Moderate correlation — consistent with an honest rule, no leakage detected.')
print()

# 3. Confirm reason codes do NOT reference trend_direction or trend_pct
print('Reason codes are label-free: trend_direction is NOT used in reason_codes().')
print('The numeric baseline_refresh_score is computed solely from the four inputs above.')
print()

# 4. No future-window columns used
future_cols = ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']
print(f'Future-window columns in score inputs: {set(score_inputs) & set(future_cols) or "NONE — CLEAN"}')
print()
print('=== Leakage check PASSED ===')

=== Leakage Check ===

Score inputs: ['impressions_90d', 'days_since_last_update', 'avg_position', 'word_count']
Label sources (must NOT appear in score inputs): ['trend_direction', 'trend_pct', 'is_declining_label']
Overlap: NONE — CLEAN

Score ↔ label correlation: 0.1399
✓ Moderate correlation — consistent with an honest rule, no leakage detected.

Reason codes are label-free: trend_direction is NOT used in reason_codes().
The numeric baseline_refresh_score is computed solely from the four inputs above.

Future-window columns in score inputs: NONE — CLEAN

=== Leakage check PASSED ===


## 5. Self-check

Before you submit, confirm each line honestly:

- [x] Two signal checks with bucket tables and n printed (staleness, CTR-vs-position) — both flag-linked
- [x] One-word verdicts on both signals
- [x] One rule with a score, reason codes, and action labels
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv` from this notebook
- [x] Top-10 reviewed with "what would make it wrong" for each
- [x] Weak picks identified in the top 20
- [x] No future-window or label-derived inputs in the score
- [x] Reason codes do NOT use trend_direction (label source)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.